# nano-dsv4.1f — prepare staged training corpora

CPU/data-preparation notebook. It finds the frozen nano tokenizer attached under `/kaggle/input`,
checks out the corpus-curriculum branch, builds the early/middle/late-mid packed LM shards, and
builds the separate integer-reasoning-effort SFT JSONL.

The default 10k-step / 4K curriculum writes 10,500 packed rows (5% shuffle headroom).


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/xiayicheng3-code/nano-dsv4.1f.git"
REPO_REF = "codex/corpus-curriculum"
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "nano-dsv4.1f"
LM_OUT = WORK / "nano-dsv41f-corpus"
SFT_OUT = WORK / "nano-dsv41f-reasoning-sft"

BUILD_LM = True
BUILD_SFT = True
TOTAL_STEPS = 10_000
SEQ_LEN = 4096
QUERY_BUDGET = 128
Q_THRESHOLD = 640
SFT_TARGET_TOKENS = 4_000_000
SEED = 1701

candidates = sorted(Path("/kaggle/input").rglob("tokenizer.json"))
preferred = [p for p in candidates if "nano-dsv41f" in str(p).lower() or "nano_dsv41f" in str(p).lower()]
if not candidates:
    raise FileNotFoundError("No tokenizer.json found under /kaggle/input. Attach the frozen nano tokenizer dataset first.")
TOKENIZER_PATH = (preferred or candidates)[0]
print("tokenizer:", TOKENIZER_PATH)
print("other tokenizer candidates:", [str(p) for p in candidates[:10]])


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[data]"],
    check=True,
)
print("repo commit:", commit)


In [ ]:
if BUILD_LM:
    if LM_OUT.exists():
        shutil.rmtree(LM_OUT)
    subprocess.run(
        [
            sys.executable, str(REPO_DIR / "scripts/prepare_corpus.py"),
            "--tokenizer", str(TOKENIZER_PATH),
            "--output-dir", str(LM_OUT),
            "--total-steps", str(TOTAL_STEPS),
            "--seq-len", str(SEQ_LEN),
            "--query-budget", str(QUERY_BUDGET),
            "--q-threshold", str(Q_THRESHOLD),
            "--seed", str(SEED),
        ],
        check=True,
    )


In [ ]:
if BUILD_SFT:
    if SFT_OUT.exists():
        shutil.rmtree(SFT_OUT)
    subprocess.run(
        [
            sys.executable, str(REPO_DIR / "scripts/prepare_reasoning_sft.py"),
            "--tokenizer", str(TOKENIZER_PATH),
            "--output-dir", str(SFT_OUT),
            "--target-tokens", str(SFT_TARGET_TOKENS),
            "--max-tokens", str(SEQ_LEN),
            "--seed", str(SEED),
        ],
        check=True,
    )


In [ ]:
if BUILD_LM:
    curriculum = json.loads((LM_OUT / "curriculum_manifest.json").read_text())
    print("\nLM curriculum:")
    print(json.dumps(curriculum, indent=2))
    for phase in ("early", "middle", "late_mid"):
        manifest = json.loads((LM_OUT / phase / "manifest.json").read_text())
        print("\n", phase)
        print(" rows:", manifest["packed"]["rows"])
        print(" real-token utilization:", round(manifest["packed"]["real_token_utilization"], 4))
        print(" source weights:", {
            k: round(v, 4)
            for k, v in manifest["phase"]["source_weights_actual_real_tokens"].items()
        })
        print(" mean Q-budget utilization:", round(manifest["query_packing"]["mean_budget_utilization"], 4))
        print(" expected Q density:", [
            round(x, 6) for x in manifest["query_packing"]["expected_q_density"]
        ])

if BUILD_SFT:
    sft = json.loads((SFT_OUT / "manifest.json").read_text())
    print("\nSFT:")
    print(" records:", sft["output"]["records"])
    print(" rendered tokens:", sft["output"]["rendered_tokens"])
    print(" effort values covered:", sft["reasoning_effort"]["covered_values"])
    print(" missing efforts:", sft["reasoning_effort"]["missing_values"])


## Save the outputs

After the run, create Kaggle Dataset versions from:

- `/kaggle/working/nano-dsv41f-corpus`
- `/kaggle/working/nano-dsv41f-reasoning-sft`

Keep the generated manifests with the shards. They record the tokenizer hash, source mix,
packing/Q diagnostics, and SFT effort coverage.
